In [ ]:
import csv


def mostrar_integrantes():
    print("=" * 70)
    print("PAQUETE FINAL - MÉTODOS NUMÉRICOS")
    print("Programa 3: Aproximación polinomial")
    print("Módulo: Ajuste por Spline Cúbico Natural")
    print("-" * 70)
    print("Integrantes del equipo:")
    print("1. Williams Islas Hernández")
    print("2. Luis Alberto Avendaño Zarza")
    print("=" * 70)
    print()


def menu_principal():
    while True:
        mostrar_integrantes()
        print("MENÚ PRINCIPAL")
        print("1. Ajuste por Spline Cúbico Natural")
        print("2. Salir")

        opcion = input("Seleccione una opción: ").strip()

        if opcion == "1":
            modulo_spline_cubico()
        elif opcion == "2":
            print("\nSaliendo del programa...")
            break
        else:
            print("\nOpción no válida. Intente de nuevo.\n")


def leer_entero_positivo(mensaje):
    while True:
        try:
            valor = int(input(mensaje))
            if valor <= 0:
                print("Error: debe ingresar un entero positivo.")
            else:
                return valor
        except ValueError:
            print("Error: entrada inválida. Ingrese un número entero.")


def leer_entero_no_negativo(mensaje):
    while True:
        try:
            valor = int(input(mensaje))
            if valor < 0:
                print("Error: debe ingresar un entero mayor o igual a cero.")
            else:
                return valor
        except ValueError:
            print("Error: entrada inválida. Ingrese un número entero.")


def leer_real(mensaje):
    while True:
        try:
            return float(input(mensaje))
        except ValueError:
            print("Error: entrada inválida. Ingrese un número real.")


def respuesta_si_no(mensaje):
    while True:
        r = input(mensaje).strip().upper()
        if r in ("S", "SI", "SÍ"):
            return True
        elif r in ("N", "NO"):
            return False
        else:
            print("Respuesta inválida. Escriba S o N.")


def imprimir_tabla_puntos(x, y):
    print("\nTabla de valores:")
    print("{:<8} {:<15} {:<15}".format("Fila", "xi", "yi"))
    print("-" * 42)

    for i in range(len(x)):
        print("{:<8} {:<15.8f} {:<15.8f}".format(i, x[i], y[i]))

    print()


def ordenar_puntos(x, y):
    pares = list(zip(x, y))
    pares.sort(key=lambda par: par[0])

    x_ordenado = [par[0] for par in pares]
    y_ordenado = [par[1] for par in pares]

    return x_ordenado, y_ordenado


def hay_x_repetidos(x, tolerancia=1e-12):
    for i in range(len(x) - 1):
        if abs(x[i + 1] - x[i]) < tolerancia:
            return True
    return False


def obtener_errores_tabla_spline(x, y):
    errores = []

    if x is None or y is None:
        errores.append("La tabla no existe.")
        return errores

    if len(x) != len(y):
        errores.append("Las listas x e y no tienen la misma longitud.")

    if len(x) < 3:
        errores.append("Se requieren al menos 3 puntos para construir un spline cúbico.")

    ordenados = all(x[i] <= x[i + 1] for i in range(len(x) - 1))
    if not ordenados:
        errores.append("Los valores de x no están ordenados.")

    if hay_x_repetidos(x):
        errores.append("Hay valores de x repetidos.")

    return errores


def leer_tabla_manual_spline():
    n = leer_entero_positivo("Ingrese el número de puntos en la tabla: ")

    while n < 3:
        print("Se requieren al menos 3 puntos para construir un spline cúbico.")
        n = leer_entero_positivo("Ingrese el número de puntos en la tabla: ")

    x = []
    y = []

    print("\nCapture los puntos (xi, yi):")
    for i in range(n):
        xi = leer_real(f"x[{i}] = ")
        yi = leer_real(f"y[{i}] = ")
        x.append(xi)
        y.append(yi)

    while True:
        imprimir_tabla_puntos(x, y)

        if respuesta_si_no("¿Son correctos los datos? (S/N): "):
            break

        fila = leer_entero_no_negativo("Ingrese la fila a corregir: ")
        while fila < 0 or fila >= n:
            print("Error: fila fuera de rango.")
            fila = leer_entero_no_negativo("Ingrese la fila a corregir: ")

        x[fila] = leer_real(f"Nuevo x[{fila}] = ")
        y[fila] = leer_real(f"Nuevo y[{fila}] = ")

    return x, y


def leer_tabla_csv_spline():
    print("\nLectura desde archivo CSV")
    print("El archivo debe tener dos columnas: x,y")
    print("Puede tener encabezado o no tenerlo.")
    print("Use punto decimal, por ejemplo: 2.65")
    print("\nEjemplo en Colab: datos_spline.csv")
    print("Si está en otra carpeta, use una ruta como: /content/datos_spline.csv")

    ruta = input("Ingrese el nombre o ruta del archivo CSV: ").strip()

    if ruta == "":
        print("No se ingresó ningún archivo.")
        return None, None

    ruta = ruta.strip('"').strip("'")

    x = []
    y = []

    try:
        with open(ruta, "r", encoding="utf-8-sig", newline="") as archivo:
            muestra = archivo.read(1024)
            archivo.seek(0)

            try:
                dialecto = csv.Sniffer().sniff(muestra, delimiters=",;")
            except csv.Error:
                dialecto = csv.excel

            lector = csv.reader(archivo, dialecto)

            numero_fila = 0
            encabezado_revisado = False

            for fila in lector:
                numero_fila += 1

                if len(fila) == 0:
                    continue

                fila = [dato.strip() for dato in fila]

                if len(fila) < 2:
                    print(f"Error en la fila {numero_fila}: se esperaban dos columnas.")
                    return None, None

                if not encabezado_revisado:
                    encabezado_revisado = True

                    try:
                        xi = float(fila[0])
                        yi = float(fila[1])
                    except ValueError:
                        continue

                    x.append(xi)
                    y.append(yi)
                    continue

                try:
                    xi = float(fila[0])
                    yi = float(fila[1])
                except ValueError:
                    print(f"Error en la fila {numero_fila}: los datos deben ser numéricos.")
                    return None, None

                x.append(xi)
                y.append(yi)

    except FileNotFoundError:
        print("Error: no se encontró el archivo.")
        return None, None
    except PermissionError:
        print("Error: no se tiene permiso para abrir el archivo.")
        return None, None
    except OSError:
        print("Error: no se pudo abrir el archivo.")
        return None, None

    if len(x) == 0:
        print("El archivo no contiene datos válidos.")
        return None, None

    print("\nDatos leídos desde el archivo CSV:")
    imprimir_tabla_puntos(x, y)

    if respuesta_si_no("¿Desea corregir algún dato leído? (S/N): "):
        while True:
            fila = leer_entero_no_negativo("Ingrese la fila a corregir: ")

            while fila < 0 or fila >= len(x):
                print("Error: fila fuera de rango.")
                fila = leer_entero_no_negativo("Ingrese la fila a corregir: ")

            x[fila] = leer_real(f"Nuevo x[{fila}] = ")
            y[fila] = leer_real(f"Nuevo y[{fila}] = ")

            imprimir_tabla_puntos(x, y)

            if respuesta_si_no("¿Ya son correctos los datos? (S/N): "):
                break

    return x, y


def cargar_ejemplo_prueba():
    x = [2.65, 3.24, 4.19, 5.20, 5.95, 6.82, 8.56]
    y = [2.93, 3.98, 4.66, 4.90, 5.77, 6.14, 5.62]

    print("\nSe cargó un ejemplo de prueba.")
    imprimir_tabla_puntos(x, y)

    return x, y


def obtener_tabla_spline():
    while True:
        print("\n" + "-" * 70)
        print("ENTRADA DE DATOS PARA SPLINE CÚBICO")
        print("-" * 70)
        print("1. Capturar datos manualmente")
        print("2. Cargar datos desde archivo CSV")
        print("3. Usar ejemplo de prueba")
        print("4. Regresar al menú principal")

        opcion = input("Seleccione una opción: ").strip()

        if opcion == "1":
            return leer_tabla_manual_spline()
        elif opcion == "2":
            return leer_tabla_csv_spline()
        elif opcion == "3":
            return cargar_ejemplo_prueba()
        elif opcion == "4":
            return None, None
        else:
            print("\nOpción no válida. Intente de nuevo.")


def elegir_modo_desarrollo():
    print("\nModo de impresión:")
    print("1. Desarrollo completo")
    print("2. Solo resultados principales")

    opcion = input("Seleccione una opción: ").strip()

    while opcion not in ("1", "2"):
        print("Opción no válida.")
        opcion = input("Seleccione una opción: ").strip()

    return opcion == "1"


def calcular_hi(x):
    h = []

    for i in range(len(x) - 1):
        h.append(x[i + 1] - x[i])

    return h


def calcular_diferencias_divididas(x, y, h):
    diferencias = []

    for i in range(len(h)):
        diferencias.append((y[i + 1] - y[i]) / h[i])

    return diferencias


def imprimir_calculos_hi(x, h):
    print("\nCálculo de los valores h_i:")
    print("h_i = x[i+1] - x[i]\n")

    for i in range(len(h)):
        print(f"h[{i}] = {x[i + 1]:.8f} - {x[i]:.8f} = {h[i]:.8f}")

    print()


def imprimir_calculos_diferencias(x, y, h, diferencias):
    print("\nCálculo de diferencias divididas:")
    print("f[x_i, x_{i+1}] = (y[i+1] - y[i]) / h_i\n")

    for i in range(len(diferencias)):
        print(
            f"f[x{i}, x{i + 1}] = "
            f"({y[i + 1]:.8f} - {y[i]:.8f}) / {h[i]:.8f} "
            f"= {diferencias[i]:.8f}"
        )

    print()


def imprimir_tabla_spline(x, y, h, diferencias):
    print("\nTabla para spline cúbico:")
    print("{:<8} {:<15} {:<15} {:<15} {:<18}".format(
        "i", "xi", "yi", "hi", "f[xi,xi+1]"
    ))
    print("-" * 75)

    for i in range(len(x)):
        if i < len(h):
            print("{:<8} {:<15.8f} {:<15.8f} {:<15.8f} {:<18.8f}".format(
                i, x[i], y[i], h[i], diferencias[i]
            ))
        else:
            print("{:<8} {:<15.8f} {:<15.8f} {:<15} {:<18}".format(
                i, x[i], y[i], "-", "-"
            ))

    print()


def imprimir_ecuaciones_spline(h, diferencias):
    n = len(h) + 1

    print("\nConstrucción del sistema de ecuaciones:")
    print("Para spline cúbico natural se usa:")
    print("S0 = 0 y Sn = 0")
    print()
    print("h[i-1]S[i-1] + 2(h[i-1] + h[i])S[i] + h[i]S[i+1]")
    print("= 6(f[i] - f[i-1])")
    print()

    for i in range(1, n - 1):
        lado_derecho = 6 * (diferencias[i] - diferencias[i - 1])

        print(f"Para i = {i}:")
        print(
            f"{h[i - 1]:.8f}S{i - 1} + "
            f"2({h[i - 1]:.8f} + {h[i]:.8f})S{i} + "
            f"{h[i]:.8f}S{i + 1} "
            f"= 6({diferencias[i]:.8f} - {diferencias[i - 1]:.8f})"
        )
        print(
            f"{h[i - 1]:.8f}S{i - 1} + "
            f"{2 * (h[i - 1] + h[i]):.8f}S{i} + "
            f"{h[i]:.8f}S{i + 1} "
            f"= {lado_derecho:.8f}"
        )
        print()


def construir_sistema_spline(h, diferencias):
    n = len(h) + 1
    cantidad_incognitas = n - 2

    A = [[0.0 for _ in range(cantidad_incognitas)] for _ in range(cantidad_incognitas)]
    b = [0.0 for _ in range(cantidad_incognitas)]

    for i in range(1, n - 1):
        fila = i - 1

        if i - 1 >= 1:
            A[fila][fila - 1] = h[i - 1]

        A[fila][fila] = 2 * (h[i - 1] + h[i])

        if i + 1 <= n - 2:
            A[fila][fila + 1] = h[i]

        b[fila] = 6 * (diferencias[i] - diferencias[i - 1])

    return A, b


def imprimir_matriz_sistema(A, b):
    print("\nSistema matricial:")
    print("A * S = b")
    print()

    cantidad_incognitas = len(A)

    encabezado = ""
    for i in range(1, cantidad_incognitas + 1):
        encabezado += f"{'S' + str(i):>14}"
    encabezado += f"{'b':>14}"

    print(encabezado)
    print("-" * (14 * (cantidad_incognitas + 1)))

    for i in range(cantidad_incognitas):
        for j in range(cantidad_incognitas):
            print(f"{A[i][j]:>14.8f}", end="")
        print(f"{b[i]:>14.8f}")

    print()


def resolver_sistema_gauss(A, b):
    n = len(A)

    matriz = []
    for i in range(n):
        fila = A[i][:]
        fila.append(b[i])
        matriz.append(fila)

    for col in range(n):
        fila_pivote = col

        for fila in range(col + 1, n):
            if abs(matriz[fila][col]) > abs(matriz[fila_pivote][col]):
                fila_pivote = fila

        if abs(matriz[fila_pivote][col]) < 1e-12:
            raise ValueError("El sistema no se puede resolver porque tiene pivote cero.")

        if fila_pivote != col:
            matriz[col], matriz[fila_pivote] = matriz[fila_pivote], matriz[col]

        pivote = matriz[col][col]

        for j in range(col, n + 1):
            matriz[col][j] = matriz[col][j] / pivote

        for fila in range(n):
            if fila != col:
                factor = matriz[fila][col]

                for j in range(col, n + 1):
                    matriz[fila][j] = matriz[fila][j] - factor * matriz[col][j]

    solucion = []

    for i in range(n):
        solucion.append(matriz[i][n])

    return solucion


def obtener_valores_S(A, b, cantidad_puntos):
    solucion_interna = resolver_sistema_gauss(A, b)

    S = [0.0 for _ in range(cantidad_puntos)]

    for i in range(1, cantidad_puntos - 1):
        S[i] = solucion_interna[i - 1]

    return S


def imprimir_valores_S(S):
    print("\nValores obtenidos de S_i:")
    print("{:<8} {:<15}".format("i", "S_i"))
    print("-" * 25)

    for i in range(len(S)):
        print("{:<8} {:<15.8f}".format(i, S[i]))

    print()


def imprimir_formulas_coeficientes():
    print("\nFórmulas para los coeficientes:")
    print("a_i = (S[i+1] - S[i]) / (6h_i)")
    print("b_i = S[i] / 2")
    print("c_i = f[x_i,x_{i+1}] - (h_i/6)(2S[i] + S[i+1])")
    print("d_i = y_i")
    print()


def calcular_coeficientes_spline(x, y, h, diferencias, S):
    coeficientes = []

    for i in range(len(h)):
        ai = (S[i + 1] - S[i]) / (6 * h[i])
        bi = S[i] / 2
        ci = diferencias[i] - (h[i] / 6) * (2 * S[i] + S[i + 1])
        di = y[i]

        coeficientes.append({
            "i": i,
            "a": ai,
            "b": bi,
            "c": ci,
            "d": di,
            "xi": x[i],
            "xf": x[i + 1]
        })

    return coeficientes


def imprimir_calculos_coeficientes(x, y, h, diferencias, S):
    print("\nCálculo de coeficientes por intervalo:")

    for i in range(len(h)):
        ai = (S[i + 1] - S[i]) / (6 * h[i])
        bi = S[i] / 2
        ci = diferencias[i] - (h[i] / 6) * (2 * S[i] + S[i + 1])
        di = y[i]

        print(f"\nIntervalo {i}: [{x[i]:.8f}, {x[i + 1]:.8f}]")

        print(
            f"a[{i}] = ({S[i + 1]:.8f} - {S[i]:.8f}) / "
            f"(6({h[i]:.8f})) = {ai:.8f}"
        )

        print(f"b[{i}] = {S[i]:.8f} / 2 = {bi:.8f}")

        print(
            f"c[{i}] = {diferencias[i]:.8f} - "
            f"({h[i]:.8f}/6)(2({S[i]:.8f}) + {S[i + 1]:.8f}) "
            f"= {ci:.8f}"
        )

        print(f"d[{i}] = y[{i}] = {di:.8f}")

    print()


def imprimir_tabla_coeficientes(coeficientes):
    print("\nTabla de coeficientes del spline cúbico:")
    print("{:<8} {:<15} {:<15} {:<15} {:<15} {:<20}".format(
        "i", "a_i", "b_i", "c_i", "d_i", "Intervalo"
    ))
    print("-" * 95)

    for dato in coeficientes:
        intervalo = f"[{dato['xi']:.8f}, {dato['xf']:.8f}]"

        print("{:<8} {:<15.8f} {:<15.8f} {:<15.8f} {:<15.8f} {:<20}".format(
            dato["i"], dato["a"], dato["b"], dato["c"], dato["d"], intervalo
        ))

    print()


def texto_x_menos_xi(xi):
    if xi < 0:
        return f"(x + {abs(xi):.8f})"
    return f"(x - {xi:.8f})"


def formatear_termino(coeficiente, texto_variable, es_primer_termino=False):
    if abs(coeficiente) < 1e-12:
        return ""

    if es_primer_termino:
        signo = "-" if coeficiente < 0 else ""
    else:
        signo = " - " if coeficiente < 0 else " + "

    return signo + f"{abs(coeficiente):.8f}{texto_variable}"


def construir_polinomio_texto(dato):
    xi = dato["xi"]
    a = dato["a"]
    b = dato["b"]
    c = dato["c"]
    d = dato["d"]

    base = texto_x_menos_xi(xi)

    partes = []

    termino_a = formatear_termino(a, f"{base}^3", True)
    if termino_a != "":
        partes.append(termino_a)

    termino_b = formatear_termino(b, f"{base}^2", len(partes) == 0)
    if termino_b != "":
        partes.append(termino_b)

    termino_c = formatear_termino(c, f"{base}", len(partes) == 0)
    if termino_c != "":
        partes.append(termino_c)

    if len(partes) == 0:
        partes.append(f"{d:.8f}")
    else:
        if d < 0:
            partes.append(f" - {abs(d):.8f}")
        else:
            partes.append(f" + {d:.8f}")

    return "".join(partes)


def imprimir_forma_general_polinomio():
    print("\nForma general de cada polinomio:")
    print("g_i(x) = a_i(x - x_i)^3 + b_i(x - x_i)^2 + c_i(x - x_i) + d_i")
    print()


def imprimir_polinomios_spline(coeficientes):
    print("\nPolinomios del spline cúbico natural:\n")

    for dato in coeficientes:
        i = dato["i"]
        xi = dato["xi"]
        xf = dato["xf"]

        polinomio = construir_polinomio_texto(dato)

        print(f"g{i}(x) = {polinomio}")
        print(f"Intervalo: [{xi:.8f}, {xf:.8f}]")
        print()


def evaluar_polinomio_spline(dato, valor_x):
    xi = dato["xi"]
    a = dato["a"]
    b = dato["b"]
    c = dato["c"]
    d = dato["d"]

    return (
        a * (valor_x - xi) ** 3
        + b * (valor_x - xi) ** 2
        + c * (valor_x - xi)
        + d
    )


def generar_puntos_intervalo(inicio, fin, cantidad=60):
    puntos = []

    if cantidad <= 1:
        return [inicio, fin]

    paso = (fin - inicio) / (cantidad - 1)

    for i in range(cantidad):
        puntos.append(inicio + i * paso)

    return puntos


def graficar_spline(x, y, coeficientes):
    try:
        import matplotlib.pyplot as plt
    except ImportError:
        print("\nNo se pudo cargar matplotlib.")
        print("Instale la librería con: pip install matplotlib")
        return

    valores_x = []
    valores_y = []

    for dato in coeficientes:
        xi = dato["xi"]
        xf = dato["xf"]

        puntos_intervalo = generar_puntos_intervalo(xi, xf, 60)

        for valor_x in puntos_intervalo:
            valor_y = evaluar_polinomio_spline(dato, valor_x)
            valores_x.append(valor_x)
            valores_y.append(valor_y)

    plt.figure(figsize=(9, 6))

    plt.plot(valores_x, valores_y, label="Spline cúbico natural")
    plt.scatter(x, y, label="Puntos originales")

    for i in range(len(x)):
        plt.text(x[i], y[i], f"  P{i}")

    plt.title("Ajuste por spline cúbico natural")
    plt.xlabel("x")
    plt.ylabel("y")
    plt.grid(True)
    plt.legend()

    if respuesta_si_no("¿Desea guardar la gráfica como imagen? (S/N): "):
        nombre = input("Nombre del archivo, por ejemplo grafica_spline.png: ").strip()

        if nombre == "":
            nombre = "grafica_spline.png"

        if not nombre.lower().endswith((".png", ".jpg", ".jpeg", ".pdf")):
            nombre += ".png"

        plt.savefig(nombre, dpi=300, bbox_inches="tight")
        print(f"Gráfica guardada como: {nombre}")

        try:
            from google.colab import files
            files.download(nombre)
        except ImportError:
            pass

    plt.show()


def validar_tabla_para_spline(x, y):
    while True:
        ordenados = all(x[i] <= x[i + 1] for i in range(len(x) - 1))

        if not ordenados:
            print("\nLos datos no estaban ordenados. Se ordenarán automáticamente.")
            x, y = ordenar_puntos(x, y)

        imprimir_tabla_puntos(x, y)

        errores = obtener_errores_tabla_spline(x, y)

        if not errores:
            print("La tabla es válida para construir el spline cúbico natural.\n")
            return x, y

        print("La tabla tiene los siguientes problemas:")
        for error in errores:
            print("-", error)
        print()

        if len(x) < 3:
            print("No se puede continuar con menos de 3 puntos.")
            return None, None

        if not respuesta_si_no("¿Desea corregir una fila? (S/N): "):
            return None, None

        fila = leer_entero_no_negativo("Ingrese la fila que desea corregir: ")
        while fila < 0 or fila >= len(x):
            print("Error: fila fuera de rango.")
            fila = leer_entero_no_negativo("Ingrese la fila que desea corregir: ")

        x[fila] = leer_real(f"Nuevo x[{fila}] = ")
        y[fila] = leer_real(f"Nuevo y[{fila}] = ")


def resolver_spline_con_tabla(x, y, desarrollo_completo):
    print("\n" + "-" * 70)
    print("CÁLCULO DE h_i Y DIFERENCIAS DIVIDIDAS")
    print("-" * 70)

    h = calcular_hi(x)
    diferencias = calcular_diferencias_divididas(x, y, h)

    if desarrollo_completo:
        imprimir_calculos_hi(x, h)
        imprimir_calculos_diferencias(x, y, h, diferencias)

    imprimir_tabla_spline(x, y, h, diferencias)

    print("\n" + "-" * 70)
    print("SISTEMA PARA OBTENER LOS VALORES S_i")
    print("-" * 70)

    if desarrollo_completo:
        imprimir_ecuaciones_spline(h, diferencias)

    A, b = construir_sistema_spline(h, diferencias)
    imprimir_matriz_sistema(A, b)

    try:
        S = obtener_valores_S(A, b, len(x))
        imprimir_valores_S(S)
    except ValueError as error:
        print("Error al resolver el sistema:")
        print(error)
        return

    print("\n" + "-" * 70)
    print("CÁLCULO DE COEFICIENTES a_i, b_i, c_i, d_i")
    print("-" * 70)

    if desarrollo_completo:
        imprimir_formulas_coeficientes()

    coeficientes = calcular_coeficientes_spline(x, y, h, diferencias, S)

    if desarrollo_completo:
        imprimir_calculos_coeficientes(x, y, h, diferencias, S)

    imprimir_tabla_coeficientes(coeficientes)

    print("\n" + "-" * 70)
    print("POLINOMIOS DEL SPLINE CÚBICO NATURAL")
    print("-" * 70)

    imprimir_forma_general_polinomio()
    imprimir_polinomios_spline(coeficientes)

    print("\n" + "-" * 70)
    print("GRÁFICA DEL SPLINE CÚBICO NATURAL")
    print("-" * 70)

    if respuesta_si_no("¿Desea mostrar la gráfica del spline? (S/N): "):
        graficar_spline(x, y, coeficientes)
    else:
        print("Se omitió la gráfica.")

    print("\nEl spline cúbico natural fue calculado correctamente.")


def modulo_spline_cubico():
    while True:
        print("\n" + "=" * 70)
        print("MÓDULO: AJUSTE POR SPLINE CÚBICO NATURAL")
        print("=" * 70)

        x, y = obtener_tabla_spline()

        if x is None or y is None:
            print()
            return

        x, y = validar_tabla_para_spline(x, y)

        if x is None or y is None:
            print()
            return

        desarrollo_completo = elegir_modo_desarrollo()

        resolver_spline_con_tabla(x, y, desarrollo_completo)

        print("\n¿Qué desea hacer ahora?")
        print("1. Realizar otro ajuste")
        print("2. Regresar al menú principal")

        opcion = input("Seleccione una opción: ").strip()

        while opcion not in ("1", "2"):
            print("Opción no válida.")
            opcion = input("Seleccione una opción: ").strip()

        if opcion == "1":
            continue
        else:
            print()
            return


if __name__ == "__main__":
    menu_principal()

PAQUETE FINAL - MÉTODOS NUMÉRICOS
Programa 3: Aproximación polinomial
Módulo: Ajuste por Spline Cúbico Natural
----------------------------------------------------------------------
Integrantes del equipo:
1. Williams Islas Hernández
2. Luis Alberto Avendaño Zarza

MENÚ PRINCIPAL
1. Ajuste por Spline Cúbico Natural
2. Salir
